In [1]:
# 1. LECTURA Y PREPROCESAMIENTO DEL DATASET

# 1.1. Importación de librerías base
import pandas as pd
import numpy as np
import os
from datetime import datetime

# 1.2. Definición de rutas Ruta local al archivo CSV
RUTA_DATOS = r"C:\Users\dreategu\OneDrive - Grupo EPM\Documentos\Python\PRONOSTICOS\TRANSFORMER-TFT\dataset_tft_colab.csv"

# 1.3. Lectura del dataset
df = pd.read_csv(RUTA_DATOS, sep=",", encoding="utf-8-sig")

# 1.4. Conversión de FECHA a tipo datetime
df['FECHA'] = pd.to_datetime(df['FECHA'], errors='coerce')

# 1.5. Eliminación de la columna innecesaria "PANDEMIA"
if 'PANDEMIA' in df.columns:
    df = df.drop(columns=['PANDEMIA'])

# 1.6. Conversión de tipos numéricos
columnas_numericas = [col for col in df.columns if col not in ['AGENTE', 'FECHA']]
df[columnas_numericas] = df[columnas_numericas].apply(pd.to_numeric, errors='coerce')

# 1.7. Ordenamiento y verificación
df = df.sort_values(by=['AGENTE', 'FECHA']).reset_index(drop=True)

# 1.8Muestra resumen general
print("Estructura del dataset:")
print(df.dtypes)
print("\nRango temporal:", df['FECHA'].min(), "→", df['FECHA'].max())
print("\nColumnas:", list(df.columns))
df


Estructura del dataset:
AGENTE                                 object
FECHA                          datetime64[ns]
FECHA_IDX                             float64
gen_mes                               float64
comp_bols_mes                         float64
comp_cont_mes                         float64
comp_recon_mes                        float64
demanda_real_mes                      float64
comp_cont_reg_mes                     float64
vent_cont_SICEP_mes                   float64
vent_cont_no_reg_mes                  float64
vent_cont_ener_mes                    float64
precio_bolsa_mes                      float64
precio_prom_cont_reg_mes              float64
precio_prom_cont_no_reg_mes           float64
pandemia                              float64
ONI                                   float64
Demanda_UPME                            int64
EXP_BOLSA                             float64
dtype: object

Rango temporal: 2010-01-01 00:00:00 → 2026-12-01 00:00:00

Columnas: ['AGENTE', 'FECHA'

,AGENTE,FECHA,FECHA_IDX,gen_mes,comp_bols_mes,comp_cont_mes,comp_recon_mes,demanda_real_mes,comp_cont_reg_mes,vent_cont_SICEP_mes,vent_cont_no_reg_mes,vent_cont_ener_mes,precio_bolsa_mes,precio_prom_cont_reg_mes,precio_prom_cont_no_reg_mes,pandemia,ONI,Demanda_UPME,EXP_BOLSA
0,BTGC,2010-01-01,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5,4577,0.0
1,BTGC,2010-02-01,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.2,4409,0.0
2,BTGC,2010-03-01,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.8,4890,0.0
3,BTGC,2010-04-01,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,4611,0.0
4,BTGC,2010-05-01,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.2,4788,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6319,TRMG,2026-08-01,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,7383,0.0
6320,TRMG,2026-09-01,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7153,0.0
6321,TRMG,2026-10-01,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7281,0.0
6322,TRMG,2026-11-01,11.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7088,0.0


In [2]:
# 2. DEFINICIÓN DE FECHAS DE CORTE Y CONFIGURACIÓN

# 2.1. Definición de fechas clave

# Estas variables controlan el periodo de entrenamiento, test y proyección.
# Se pueden modificar fácilmente para nuevos experimentos.
fecha_inicio          = datetime(2010, 1, 1)
fecha_fin_hist        = datetime(2025, 9, 30)
fecha_fin_forecast    = datetime(2026, 12, 31)
fecha_corte_train     = datetime(2024, 12, 31)  # ejemplo de corte de entrenamiento

# 2.2. Clasificación de variables
# Variable objetivo
var_objetivo = 'EXP_BOLSA'

# Variables exógenas con proyección
vars_exogenas = ['ONI', 'Demanda_UPME', 'pandemia']

# Variables endógenas internas (todas las demás excepto AGENTE, FECHA, FECHA_IDX y las anteriores)
vars_exogenas_aux = vars_exogenas + ['AGENTE', 'FECHA', 'FECHA_IDX', var_objetivo]
vars_endogenas = [col for col in df.columns if col not in vars_exogenas_aux]

# 2.3. Particiones temporales
# Entrenamiento: hasta fecha_corte_train
# Validación/Test: posterior a fecha_corte_train hasta fin histórico
df_train = df[df['FECHA'] <= fecha_corte_train].copy()
df_test  = df[(df['FECHA'] > fecha_corte_train) & (df['FECHA'] <= fecha_fin_hist)].copy()

print(f"Período de entrenamiento: {df_train['FECHA'].min().date()} → {df_train['FECHA'].max().date()}")
print(f"Período de prueba: {df_test['FECHA'].min().date()} → {df_test['FECHA'].max().date()}")

# 2.4. Visualización rápida del tamaño por agente
conteo_por_agente = df.groupby('AGENTE').size().reset_index(name='n_registros')
print("\nRegistros por agente:")
print(conteo_por_agente.head())

# 2.5. Confirmación de variables
print("\nVariable objetivo:", var_objetivo)
print("Variables exógenas:", vars_exogenas)
print("Variables endógenas:", vars_endogenas)


Período de entrenamiento: 2010-01-01 → 2024-12-01
Período de prueba: 2025-01-01 → 2025-09-01

Registros por agente:
  AGENTE  n_registros
0   BTGC          204
1   CHVG          204
2   CMMC          204
3   CNSC          204
4   CSIC          204

Variable objetivo: EXP_BOLSA
Variables exógenas: ['ONI', 'Demanda_UPME', 'pandemia']
Variables endógenas: ['gen_mes', 'comp_bols_mes', 'comp_cont_mes', 'comp_recon_mes', 'demanda_real_mes', 'comp_cont_reg_mes', 'vent_cont_SICEP_mes', 'vent_cont_no_reg_mes', 'vent_cont_ener_mes', 'precio_bolsa_mes', 'precio_prom_cont_reg_mes', 'precio_prom_cont_no_reg_mes']


In [3]:
# 3. Importación de librerías para modelado

import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [4]:
# 4. Definición de funciones auxiliares

# 4.1. Calcular métricas calcula MAE, RMSE, MAPE, R2 para determinar luego el mejor modelo
def calcular_metricas(y_true, y_pred):
    """
    Calcula métricas básicas de evaluación para un modelo de predicción.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-10))) * 100
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

# 4.2. Entrenamiento de modelos
def entrenar_y_evaluar_modelo(nombre_modelo, modelo, X_train, y_train, X_test, y_test):
    """
    Entrena un modelo de regresión supervisada y devuelve predicciones + métricas.
    """
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    metricas = calcular_metricas(y_test, y_pred)
    return {"modelo": nombre_modelo, "metricas": metricas, "predicciones": y_pred, "modelo_entrenado": modelo}



In [5]:
# 5. Preparación del pipeline por agente

resultados_agentes = []  # Almacena los resultados de cada agente

agentes = df_train['AGENTE'].unique()
print(f"Modelando {len(agentes)} agentes...")

Modelando 31 agentes...


In [6]:
# ======================================================
# Función optimizada para búsqueda de mejores parámetros
# ARIMA / SARIMA / SARIMAX (versión híbrida rápida)
# ======================================================
import itertools
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tqdm import tqdm  # muestra barra de progreso
warnings.filterwarnings("ignore")

def buscar_mejores_parametros(y, exog=None, estacional=False, m=12, max_p=2, max_d=1, max_q=2):
    """
    Busca combinaciones (p,d,q) y (P,D,Q,m) con menor AIC.
    - Limita el espacio de búsqueda para acelerar la ejecución.
    - Fija el componente estacional a (0,1,1,m), común en series mensuales.
    - Soporta modelos con o sin exógenas.
    """

    mejor_aic = np.inf
    mejor_orden = None
    mejor_orden_est = None

    # Definir el espacio de búsqueda reducido
    p = range(0, max_p + 1)
    d = range(0, max_d + 1)
    q = range(0, max_q + 1)
    pdq = list(itertools.product(p, d, q))

    # Fijar el componente estacional para acelerar la búsqueda
    PDQ = [(0, 1, 1, m)] if estacional else [(0, 0, 0, 0)]

    # Iterar sobre combinaciones de (p,d,q)
    for orden in tqdm(pdq, desc="Buscando mejores parámetros", leave=False):
        for orden_est in PDQ:
            try:
                modelo = SARIMAX(
                    y,
                    exog=exog,
                    order=orden,
                    seasonal_order=orden_est if estacional else (0, 0, 0, 0),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False)
                if modelo.aic < mejor_aic:
                    mejor_aic = modelo.aic
                    mejor_orden = orden
                    mejor_orden_est = orden_est
            except:
                continue

    return mejor_orden, mejor_orden_est, mejor_aic


In [8]:
# ======================================================
# BLOQUE 6 - Modelado por agente con tuning rápido y validación cruzada temporal
# ======================================================
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer
from scipy.stats import randint, uniform
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ------------------------------------------------------
# Definición de validación cruzada temporal
# ------------------------------------------------------
n_splits_val = 4   # <--- puedes cambiar esta variable manualmente
test_size_val = 12 # <--- tamaño del bloque de validación (en meses)
tscv = TimeSeriesSplit(n_splits=n_splits_val, test_size=test_size_val)

# Scorer personalizado para MAPE (menor es mejor)
def mape_scorer(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

mape_scorer_neg = make_scorer(mape_scorer, greater_is_better=False)

# ------------------------------------------------------
# Función auxiliar: validación cruzada manual promedio MAPE
# ------------------------------------------------------
def crossval_mape_promedio(modelo, X, y, splits=tscv):
    mape_scores = []
    for train_idx, val_idx in splits.split(X):
        X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
        y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]
        modelo.fit(X_train_cv, y_train_cv)
        pred_val = modelo.predict(X_val_cv)
        mape_val = mape_scorer(y_val_cv, pred_val)
        mape_scores.append(mape_val)
    return np.mean(mape_scores)

# ------------------------------------------------------
# Bucle principal
# ------------------------------------------------------
for agente in agentes:
    print(f"\nProcesando agente: {agente}")
    
    # Filtrado de datos del agente
    df_agente_train = df_train[df_train['AGENTE'] == agente].copy()
    df_agente_test  = df_test[df_test['AGENTE'] == agente].copy()

    # Selección de variables
    X_train = df_agente_train[vars_exogenas + vars_endogenas].fillna(0)
    y_train = df_agente_train[var_objetivo]
    X_test  = df_agente_test[vars_exogenas + vars_endogenas].fillna(0)
    y_test  = df_agente_test[var_objetivo]

    modelos = []

    # ======================================================
    # MODELO 1: ARIMA (univariante con búsqueda automática)
    # ======================================================
    try:
        serie_train = y_train.values
        orden_arima, _, aic_arima = buscar_mejores_parametros(serie_train, estacional=False)
        modelo_arima = sm.tsa.ARIMA(serie_train, order=orden_arima).fit()
        pred_arima = modelo_arima.forecast(steps=len(y_test))
        metricas_arima = calcular_metricas(y_test.values, pred_arima)
        modelos.append({
            "modelo": f"ARIMA{orden_arima}",
            "metricas": metricas_arima,
            "predicciones": pred_arima
        })
    except Exception as e:
        print(f"ARIMA falló para {agente}: {e}")

    # ======================================================
    # MODELO 2: SARIMA (estacional)
    # ======================================================
    try:
        orden_sarima, orden_est_sarima, aic_sarima = buscar_mejores_parametros(serie_train, estacional=True, m=12)
        modelo_sarima = SARIMAX(serie_train, order=orden_sarima, seasonal_order=orden_est_sarima).fit(disp=False)
        pred_sarima = modelo_sarima.forecast(steps=len(y_test))
        metricas_sarima = calcular_metricas(y_test.values, pred_sarima)
        modelos.append({
            "modelo": f"SARIMA{orden_sarima}-{orden_est_sarima}",
            "metricas": metricas_sarima,
            "predicciones": pred_sarima
        })
    except Exception as e:
        print(f"SARIMA falló para {agente}: {e}")

    # ======================================================
    # MODELO 3: SARIMAX (con exógenas proyectadas)
    # ======================================================
    try:
        orden_sarimax, orden_est_sarimax, aic_sarimax = buscar_mejores_parametros(
            y_train, exog=X_train[vars_exogenas], estacional=True, m=12
        )
        modelo_sarimax = SARIMAX(
            y_train,
            exog=X_train[vars_exogenas],
            order=orden_sarimax,
            seasonal_order=orden_est_sarimax,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)
        pred_sarimax = modelo_sarimax.predict(
            start=len(y_train),
            end=len(y_train) + len(y_test) - 1,
            exog=X_test[vars_exogenas]
        )
        metricas_sarimax = calcular_metricas(y_test.values, pred_sarimax)
        modelos.append({
            "modelo": f"SARIMAX{orden_sarimax}-{orden_est_sarimax}",
            "metricas": metricas_sarimax,
            "predicciones": pred_sarimax
        })
    except Exception as e:
        print(f"SARIMAX falló para {agente}: {e}")

    # ======================================================
    # MODELO 4: XGBoost con búsqueda rápida y CV temporal
    # ======================================================
    try:
        param_dist_xgb = {
            "n_estimators": randint(300, 600),
            "learning_rate": uniform(0.01, 0.1),
            "max_depth": randint(3, 6),
            "subsample": uniform(0.7, 0.3),
            "colsample_bytree": uniform(0.7, 0.3)
        }
        xgb_base = XGBRegressor(random_state=42)
        xgb_search = RandomizedSearchCV(
            xgb_base,
            param_distributions=param_dist_xgb,
            n_iter=10,  # búsqueda rápida
            scoring=mape_scorer_neg,
            cv=tscv,
            n_jobs=-1,
            random_state=42
        )
        xgb_search.fit(X_train, y_train)
        mejor_xgb = xgb_search.best_estimator_
        pred_xgb = mejor_xgb.predict(X_test)
        metricas_xgb = calcular_metricas(y_test, pred_xgb)
        modelos.append({
            "modelo": f"XGBoost_Tuned({xgb_search.best_params_})",
            "metricas": metricas_xgb,
            "predicciones": pred_xgb
        })
    except Exception as e:
        print(f"XGBoost falló para {agente}: {e}")

    # ======================================================
    # MODELO 5: Random Forest con búsqueda rápida y CV temporal
    # ======================================================
    try:
        param_dist_rf = {
            "n_estimators": randint(200, 500),
            "max_depth": randint(4, 10),
            "min_samples_split": randint(2, 8),
            "min_samples_leaf": randint(1, 5)
        }
        rf_base = RandomForestRegressor(random_state=42)
        rf_search = RandomizedSearchCV(
            rf_base,
            param_distributions=param_dist_rf,
            n_iter=10,
            scoring=mape_scorer_neg,
            cv=tscv,
            n_jobs=-1,
            random_state=42
        )
        rf_search.fit(X_train, y_train)
        mejor_rf = rf_search.best_estimator_
        pred_rf = mejor_rf.predict(X_test)
        metricas_rf = calcular_metricas(y_test, pred_rf)
        modelos.append({
            "modelo": f"RandomForest_Tuned({rf_search.best_params_})",
            "metricas": metricas_rf,
            "predicciones": pred_rf
        })
    except Exception as e:
        print(f"RandomForest falló para {agente}: {e}")

    # ======================================================
    # MODELO 6: Perceptrón Multicapa (MLP)
    # ======================================================
    try:
        mlp = MLPRegressor(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            max_iter=1000,
            random_state=42
        )
        resultado_mlp = entrenar_y_evaluar_modelo("MLP", mlp, X_train, y_train, X_test, y_test)
        modelos.append(resultado_mlp)
    except Exception as e:
        print(f"MLP falló para {agente}: {e}")

    # ======================================================
    # SELECCIÓN DEL MEJOR MODELO (menor MAPE)
    # ======================================================
    try:
        mejor = min(modelos, key=lambda m: m['metricas']['MAPE'])
        print(f"Mejor modelo para {agente}: {mejor['modelo']} | MAPE = {mejor['metricas']['MAPE']:.2f}")
        resultados_agentes.append({
            "agente": agente,
            "mejor_modelo": mejor['modelo'],
            "metricas": mejor['metricas']
        })
    except Exception as e:
        print(f"Error seleccionando mejor modelo para {agente}: {e}")



Procesando agente: BTGC


Buscando mejores parámetros:   0%|          | 0/18 [00:00<?, ?it/s]

Mejor modelo para BTGC: RandomForest_Tuned({'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 271}) | MAPE = 648138482.23

Procesando agente: CHVG


Mejor modelo para CHVG: XGBoost_Tuned({'colsample_bytree': np.float64(0.9425192044349383), 'learning_rate': np.float64(0.04046137691733707), 'max_depth': 3, 'n_estimators': 391, 'subsample': np.float64(0.8320457481218804)}) | MAPE = 5.03

Procesando agente: CMMC


Mejor modelo para CMMC: XGBoost_Tuned({'colsample_bytree': np.float64(0.7468055921327309), 'learning_rate': np.float64(0.025599452033620268), 'max_depth': 5, 'n_estimators': 387, 'subsample': np.float64(0.8001125833417065)}) | MAPE = 2.78

Procesando agente: CNSC


...

Mejor modelo para SOEC: RandomForest_Tuned({'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 4, 'n_estimators': 271}) | MAPE = 1183165.39

Procesando agente: SPRG


Mejor modelo para SPRG: XGBoost_Tuned({'colsample_bytree': np.float64(0.8574269294896713), 'learning_rate': np.float64(0.05319450186421158), 'max_depth': 3, 'n_estimators': 358, 'subsample': np.float64(0.8199582915145766)}) | MAPE = 87.99

Procesando agente: TBSG


Mejor modelo para TBSG: XGBoost_Tuned({'colsample_bytree': np.float64(0.9425192044349383), 'learning_rate': np.float64(0.04046137691733707), 'max_depth': 3, 'n_estimators': 391, 'subsample': np.float64(0.8320457481218804)}) | MAPE = 17.33

Procesando agente: TCIG


Mejor modelo para TCIG: XGBoost_Tuned({'colsample_bytree': np.float64(0.9040922615763338), 'learning_rate': np.float64(0.05504992519695431), 'max_depth': 4, 'n_estimators': 388, 'subsample': np.float64(0.9896896099223678)}) | MAPE = 78.60

Procesando agente: TRMG


Mejor modelo para TRMG: RandomForest_Tuned({'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 221}) | MAPE = 12.39


In [ ]:
# 7. Resultados globales

resultados_df = pd.DataFrame(resultados_agentes)
print("\nResumen de mejores modelos por agente:")
display(resultados_df.sort_values("mejor_modelo").reset_index(drop=True))
resultados_df


Resumen de mejores modelos por agente:


,agente,mejor_modelo,metricas
0,TCIG,ARIMA,"{'MAE': 0.005150491800812964, 'RMSE': 0.005801..."
1,TCIG,ARIMA,"{'MAE': 0.005150491800812964, 'RMSE': 0.005801..."
2,NTCG,ARIMA,"{'MAE': 0.01322896458940073, 'RMSE': 0.0163622..."
3,NTCG,ARIMA,"{'MAE': 0.01322896458940073, 'RMSE': 0.0163622..."
4,BTGC,RandomForest,"{'MAE': 0.12077158054133125, 'RMSE': 0.1410836..."
...,...,...,...
57,CNSC,XGBoost,"{'MAE': 0.016542138455637612, 'RMSE': 0.024872..."
58,CSIC,XGBoost,"{'MAE': 0.0427834461520594, 'RMSE': 0.05338173..."
59,EMIC,XGBoost,"{'MAE': 0.0032060604221242875, 'RMSE': 0.00406..."
60,ENDG,XGBoost,"{'MAE': 0.060425195065560496, 'RMSE': 0.066494..."


In [ ]:
# ==========================================================
# BLOQUE 4 - VISUALIZACIÓN Y VALIDACIÓN DE PREDICCIONES
# ==========================================================

# ----------------------------------------------------------
# 1. Importación de librerías gráficas
# ----------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 5)

# ----------------------------------------------------------
# 2. Función para graficar predicciones de un agente
# ----------------------------------------------------------
def graficar_predicciones_agente(agente, df_train, df_test, resultados_modelos, guardar=False, ruta_salida=None):
    """
    Genera una gráfica comparando valores reales vs predichos para el mejor modelo de un agente.
    
    Parámetros:
    -----------
    agente : str
        Nombre del agente a graficar.
    df_train : DataFrame
        Subconjunto de entrenamiento.
    df_test : DataFrame
        Subconjunto de prueba.
    resultados_modelos : list
        Lista de resultados del Bloque 3 (con métricas y predicciones).
    guardar : bool
        Si True, guarda la gráfica en disco.
    ruta_salida : str
        Carpeta donde guardar la imagen (si guardar=True).
    """
    try:
        # Extrae el mejor modelo y sus predicciones
        resultado_agente = next(item for item in resultados_modelos if item["agente"] == agente)
        modelo_nombre = resultado_agente["mejor_modelo"]
        print(f"\nGraficando agente: {agente} | Modelo: {modelo_nombre}")

        # Recupera los datos correspondientes al agente
        df_agente_train = df_train[df_train['AGENTE'] == agente]
        df_agente_test = df_test[df_test['AGENTE'] == agente]

        # Extrae la predicción (asumimos misma longitud que test)
        y_pred = resultado_agente.get("predicciones", None)
        if y_pred is None:
            print("No hay predicciones almacenadas para este agente.")
            return
        
        fechas_test = df_agente_test['FECHA'].values
        y_real = df_agente_test[var_objetivo].values

        # ------------------------------------------------------
        # Gráfico
        # ------------------------------------------------------
        plt.figure(figsize=(10, 5))
        plt.plot(df_agente_train['FECHA'], df_agente_train[var_objetivo], color='gray', alpha=0.6, label="Entrenamiento")
        plt.plot(fechas_test, y_real, color='black', lw=2, label="Real (Test)")
        plt.plot(fechas_test, y_pred, color='tab:blue', lw=2, linestyle='--', label=f"Predicción ({modelo_nombre})")

        plt.title(f"Agente: {agente} — Modelo: {modelo_nombre}")
        plt.xlabel("Fecha")
        plt.ylabel("EXP_BOLSA")
        plt.legend()
        plt.tight_layout()

        # Guardar imagen
        if guardar and ruta_salida:
            os.makedirs(ruta_salida, exist_ok=True)
            archivo = os.path.join(ruta_salida, f"forecast_{agente}.png")
            plt.savefig(archivo, dpi=150)
            print(f"Gráfica guardada en: {archivo}")

        plt.show()

    except StopIteration:
        print(f"No se encontró resultado para el agente {agente}.")


# ----------------------------------------------------------
# 3. Ejemplo de uso: graficar un agente específico
# ----------------------------------------------------------
agente_prueba = resultados_df['agente'].iloc[0]  # primer agente de la lista
graficar_predicciones_agente(
    agente=agente_prueba,
    df_train=df_train,
    df_test=df_test,
    resultados_modelos=resultados_agentes,
    guardar=False
)


In [ ]:
# ==========================================================
# BLOQUE 4B - EXPORTACIÓN MASIVA DE GRÁFICAS
# ==========================================================

# Ruta de salida (ajusta según tu estructura)
RUTA_SALIDA_GRAFICAS = r"C:\Users\dreategu\OneDrive - Grupo EPM\Documentos\Python\PRONOSTICOS\RESULTADOS_FORECAST"

# Recorre todos los agentes y genera una imagen por cada uno
for agente in resultados_df['agente']:
    graficar_predicciones_agente(
        agente=agente,
        df_train=df_train,
        df_test=df_test,
        resultados_modelos=resultados_agentes,
        guardar=True,
        ruta_salida=RUTA_SALIDA_GRAFICAS
    )


In [ ]:
# ==========================================================
# BLOQUE 5 - PRONÓSTICO MULTI-PASO CON VARIABLES EXÓGENAS
# ==========================================================

# ----------------------------------------------------------
# 1. Definición del horizonte de proyección (h pasos)
# ----------------------------------------------------------
# Cada "paso" equivale a un mes, ya que los datos están en frecuencia mensual
H = 36  # horizonte de 36 meses (3 años)
print(f"Horizonte de proyección definido: {H} meses")

# ----------------------------------------------------------
# 2. Generación del rango de fechas futuras
# ----------------------------------------------------------
fecha_ultimo = df['FECHA'].max()
fechas_futuras = pd.date_range(start=fecha_ultimo + pd.offsets.MonthBegin(1), periods=H, freq='MS')
print(f"Primer mes proyectado: {fechas_futuras[0].date()} — Último mes proyectado: {fechas_futuras[-1].date()}")

# ----------------------------------------------------------
# 3. Creación del DataFrame base para proyecciones
# ----------------------------------------------------------
# Incluye las variables exógenas proyectadas hasta el horizonte máximo disponible
df_futuro = df.copy()
df_futuro = df_futuro[df_futuro['FECHA'].isin(fechas_futuras) | (df_futuro['FECHA'] > fecha_ultimo)]

# Si no existen filas futuras, se construye manualmente para los agentes
if df_futuro.empty:
    agentes_unicos = df['AGENTE'].unique()
    df_futuro = pd.DataFrame({
        "AGENTE": np.repeat(agentes_unicos, H),
        "FECHA": np.tile(fechas_futuras, len(agentes_unicos))
    })

# Se incorporan las variables exógenas proyectadas (si existen)
for col in vars_exogenas:
    if col not in df_futuro.columns:
        df_futuro[col] = np.nan

# ----------------------------------------------------------
# 4. Generación de pronósticos por agente
# ----------------------------------------------------------
pronosticos_futuros = []

for agente in resultados_df['agente']:
    try:
        # Recupera el mejor modelo entrenado del agente
        resultado_agente = next(item for item in resultados_agentes if item["agente"] == agente)
        modelo_mejor = resultado_agente["modelo_entrenado"]
        modelo_nombre = resultado_agente["mejor_modelo"]

        # Filtra los datos históricos y futuros
        df_hist = df[df['AGENTE'] == agente]
        df_proj = df_futuro[df_futuro['AGENTE'] == agente]

        # Prepara las exógenas proyectadas
        X_future = df_proj[vars_exogenas + vars_endogenas].fillna(method='ffill').fillna(0)

        # Predicción futura
        y_future_pred = modelo_mejor.predict(X_future)

        # Construye DataFrame con los resultados
        df_pred = pd.DataFrame({
            "AGENTE": agente,
            "FECHA": df_proj['FECHA'].values,
            "EXP_BOLSA_PRED": y_future_pred,
            "modelo": modelo_nombre
        })
        pronosticos_futuros.append(df_pred)

        print(f"Pronóstico generado para {agente} con modelo {modelo_nombre}")

    except Exception as e:
        print(f"Error en proyección para {agente}: {e}")

# ----------------------------------------------------------
# 5. Consolidación de resultados
# ----------------------------------------------------------
df_forecasts = pd.concat(pronosticos_futuros, ignore_index=True)
print(f"\nPronósticos generados para {df_forecasts['AGENTE'].nunique()} agentes")
print(f"Total de registros proyectados: {len(df_forecasts)}")

# ----------------------------------------------------------
# 6. Visualización del forecast para un agente
# ----------------------------------------------------------
def graficar_forecast_futuro(agente):
    """
    Visualiza las proyecciones futuras junto al histórico del agente.
    """
    df_hist = df[df['AGENTE'] == agente]
    df_proj = df_forecasts[df_forecasts['AGENTE'] == agente]

    plt.figure(figsize=(10, 5))
    plt.plot(df_hist['FECHA'], df_hist[var_objetivo], color='black', label="Histórico")
    plt.plot(df_proj['FECHA'], df_proj['EXP_BOLSA_PRED'], color='tab:blue', linestyle='--', label="Proyección futura")
    plt.title(f"Agente {agente} — Proyección a {H} meses ({modelo_nombre})")
    plt.xlabel("Fecha")
    plt.ylabel("EXP_BOLSA")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Ejemplo de visualización
agente_prueba_forecast = resultados_df['agente'].iloc[0]
graficar_forecast_futuro(agente_prueba_forecast)

# ----------------------------------------------------------
# 7. Exportación de pronósticos
# ----------------------------------------------------------
ruta_salida_forecast = r"C:\Users\dreategu\OneDrive - Grupo EPM\Documentos\Python\PRONOSTICOS\RESULTADOS_FORECAST\pronosticos_EXP_BOLSA.csv"
df_forecasts.to_csv(ruta_salida_forecast, index=False, encoding='utf-8-sig')
print(f"Pronósticos guardados en: {ruta_salida_forecast}")
